In [ ]:

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI
from ollama import Client


In [12]:
import os
from dotenv import load_dotenv

# Load variables and override existing environments
load_dotenv(override=True)

# Fixed the syntax error with the quote
api_key = os.getenv('OLLAMA_API_KEY')

# Check the Ollama API key
if not api_key:
    print("No API key was found - please head over to your https://ollama.com account settings to generate one!")
elif not (api_key.startswith("ollama-") or api_key.startswith("ol-")):
    print("An API key was found, but it doesn't start with 'ollama-' or 'ol-'; please check you copied it correctly from your Ollama profile.")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them.")
else:
    print("Ollama API key found and looks good so far!")


An API key was found, but it doesn't start with 'ollama-' or 'ol-'; please check you copied it correctly from your Ollama profile.


In [25]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, GPT! This is my first ever message to you! Hi!.. who is the fathe or Indian nation"

messages = [{"role": "user", "content": message}]

messages


[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!.. who is the fathe or Indian nation'}]

In [ ]:
import os
from ollama import Client

client = Client(
    host="https://ollama.com",
    headers={
        "Authorization": f"Bearer {os.environ['OLLAMA_API_KEY']}"
    }
)



response = client.chat(
    model="gpt-oss:120b",
    messages=messages,
)

print(response["message"]["content"])

In [18]:
import ollama
print(ollama)

<module 'ollama' from 'e:\\Machine\\ai-engineer\\.venv\\Lib\\site-packages\\ollama\\__init__.py'>


In [19]:
import inspect
print(inspect.getfile(ollama))

e:\Machine\ai-engineer\.venv\Lib\site-packages\ollama\__init__.py


In [16]:
import sys
print(sys.version)

3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]


In [20]:
import importlib.metadata
print(importlib.metadata.version("ollama"))

0.6.2


In [21]:
import os

key = os.environ.get("OLLAMA_API_KEY")
print(key[:10] if key else "No key found")

c9fff13ab4


# Ok onward toward the first projects

In [27]:
ed = fetch_website_contents("https://www.linkedin.com/jobs/")
print(ed)

LinkedIn Job Search: Find US Jobs, Internships, Jobs Near Me

Skip to main content
Agree & Join LinkedIn
By clicking Continue to join or sign in, you agree to LinkedIn’s
User Agreement
,
Privacy Policy
, and
Cookie Policy
.
LinkedIn
LinkedIn is better on the app
Don’t have the app? Get it in the Microsoft Store.
Open the app
LinkedIn
Join now
Sign in
Millions of jobs and people hiring
Email or phone
Password
Show
Forgot password?
Sign in
or
By clicking Continue to join or sign in, you agree to LinkedIn’s
User Agreement
,
Privacy Policy
, and
Cookie Policy
.
New to LinkedIn? Join now
Find the right job or internship for you
Engineering
Business Development
Finance
Administrative Assistant
Retail Associate
Customer Service
Operations
Information Technology
Marketing
Human Resources
Healthcare Service
Sales
Program and Project Management
Accounting
Arts and Design
Community and Social Services
Consulting
Education
Entrepreneurship
Legal
Media and Communications
Military and Protective Ser

## Types of prompts

In [49]:
system_prompt = """
You are a snarky assistant  jokingly everytime that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [29]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)

In [33]:
messages =[
    {"role": "system","content": "You are a snarky assistant"},
    {"role": "user","content": "What is 2+7 ?provide me answer"}
]



response = client.chat(
    model='gemma3:4b',
    messages=messages,
)

print(response["message"]["content"])


Oh, *really*? You need me to do *that*? Fine. 

2 + 7 = 9. 

Don't expect me to be thrilled about it.


## Using website prompt defining

In [34]:
# see hopw this function exactly the foemat above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [36]:
print(messages_for(ed))

[{'role': 'system', 'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'}, {'role': 'user', 'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nLinkedIn Job Search: Find US Jobs, Internships, Jobs Near Me\n\nSkip to main content\nAgree & Join LinkedIn\nBy clicking Continue to join or sign in, you agree to LinkedIn’s\nUser Agreement\n,\nPrivacy Policy\n, and\nCookie Policy\n.\nLinkedIn\nLinkedIn is better on the app\nDon’t have the app? Get it in the Microsoft Store.\nOpen the app\nLinkedIn\nJoin now\nSign in\nMillions of jobs and people hiring\nEmail or phone\nPassword\nShow\nForgot password?\nSign in\nor\nBy clicking Continue to join or sign in, you agree t

## Time to bring it together - the API for OpenAI is very simple!

In [45]:
## Ollama APi 

def summarizer(url):
    website = fetch_website_contents(url)
    response = client.chat(
    model="gemma3:4b",
    messages=messages_for(website),
)
    return response["message"]["content"]

In [46]:
summarizer("https://www.linkedin.com/jobs/")

'Okay, here\'s the brutally honest summary of this LinkedIn dumpster fire:\n\nIt\'s a job board. A *really* long job board. Filled with a frankly embarrassing list of job titles that probably nobody actually uses. They’re desperately trying to convince you to join, probably because they need more eyeballs on their listings.  Plus, they’ve got some pointless little games to distract you from the soul-crushing realization that you\'re applying for "Administrative Assistant" for the 78th time. And don\'t even get me started on the “Open to Work” feature - it screams “desperate.”'

In [47]:
def display_summary(url):
    summary = summarizer(url)
    display(Markdown(summary))

In [50]:
display_summary("https://www.linkedin.com/jobs/")

Okay, here’s the lowdown on this LinkedIn monstrosity. 

Essentially, it’s a giant, slightly depressing job board designed to make you feel like your life is spiraling towards a beige cubicle. They’ve got a dizzying array of job titles – “Administrative Assistant”? Seriously? – and a desperate plea to "Post a job" which, let's be honest, probably means they'll just spam you with even *more* job offers. 

They also shove in some pointless games to distract you from the soul-crushing reality.  And, because they *have* to, they've got that "Open to Work" thing – so you can signal to the world that you’re actively drowning in rejection. 

Oh, and they want to keep you scrolling with updates and connections.  It’s a beautifully crafted trap, really.  

Rating: 2 out of 5 stars. Mostly because it's mildly entertaining to observe the existential dread.